# Chapter 10：LayerNorm Forward

实现 row-wise LayerNorm forward。每个 program 处理 `[M,N]` 的一行，mean、variance 和 affine 计算使用 fp32。

In [ ]:
from pathlib import Path
import math
import sys

ROOT = Path.cwd()
if ROOT.name.startswith("chapter_"):
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import torch.nn.functional as F
import triton
import triton.language as tl

from common.benchmark import bench
from common.check import assert_close
from common.utils import get_device, set_seed

device = get_device()
set_seed(0)

## 公式与 PyTorch reference

`mean=sum(x)/N`，`var=sum((x-mean)^2)/N`，最后乘 weight 加 bias。N 不必是 2 的幂。

In [ ]:
MAX_BLOCK_SIZE = 65_536

M, N = 1024, 513
x = torch.randn(M, N, device=device, dtype=torch.float16)
weight = torch.randn(N, device=device, dtype=torch.float16)
bias = torch.randn(N, device=device, dtype=torch.float16)
expected = F.layer_norm(x, (N,), weight, bias, 1e-5)

## Triton kernel

mask 外的值对 mean/variance 都按 0 处理；centered 的无效 lane 再次置零。

In [ ]:
@triton.jit
def layernorm_kernel(x_ptr, weight_ptr, bias_ptr, output_ptr, n_cols, eps: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    # One program normalizes one row; reductions and affine math use fp32.
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols
    x = tl.load(x_ptr + row * n_cols + cols, mask=mask, other=0.0).to(tl.float32)
    mean = tl.sum(x, axis=0) / n_cols
    centered = tl.where(mask, x - mean, 0.0)
    variance = tl.sum(centered * centered, axis=0) / n_cols
    inv_std = tl.rsqrt(variance + eps)
    weight = tl.load(weight_ptr + cols, mask=mask, other=0.0).to(tl.float32)
    bias = tl.load(bias_ptr + cols, mask=mask, other=0.0).to(tl.float32)
    output = centered * inv_std * weight + bias
    tl.store(output_ptr + row * n_cols + cols, output, mask=mask)

## Wrapper 与宽度限制

`BLOCK_SIZE=next_power_of_2(N)`。教学 kernel 对 padded width 设置 65,536 上限。

In [ ]:
def layernorm(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    if x.ndim != 2 or weight.ndim != 1 or bias.ndim != 1:
        raise ValueError("expected x[M,N], weight[N], and bias[N]")
    if weight.shape != (x.shape[1],) or bias.shape != (x.shape[1],):
        raise ValueError("weight and bias must match the last dimension of x")
    if not x.is_cuda or not weight.is_cuda or not bias.is_cuda:
        raise ValueError("x, weight, and bias must be CUDA tensors")
    if not x.is_contiguous() or not weight.is_contiguous() or not bias.is_contiguous():
        raise ValueError("x, weight, and bias must be contiguous")
    if x.dtype not in (torch.float16, torch.float32) or weight.dtype != x.dtype or bias.dtype != x.dtype:
        raise ValueError("all inputs must share fp16 or fp32 dtype")
    M, N = x.shape
    if M == 0 or N == 0:
        raise ValueError("M and N must be positive")
    block_size = triton.next_power_of_2(N)
    if block_size > MAX_BLOCK_SIZE:
        raise ValueError(f"N={N} is too large; padded width must be <= {MAX_BLOCK_SIZE}")
    output = torch.empty_like(x)
    layernorm_kernel[(M,)](x, weight, bias, output, N, eps=eps, BLOCK_SIZE=block_size, num_warps=8 if block_size >= 2048 else 4)
    return output

## Correctness 与 benchmark

fp16 输入在 fp32 中归一化，输出回到输入 dtype。

In [ ]:
actual = layernorm(x, weight, bias)
assert_close('layernorm', actual.float(), expected.float(), rtol=1e-2, atol=1e-2)
print(f'PyTorch={bench(lambda: F.layer_norm(x,(N,),weight,bias,1e-5)):.3f} ms')
print(f'Triton={bench(lambda: layernorm(x,weight,bias)):.3f} ms')

## 小结与练习

练习：测试 N=1000，确认 padded block 为 1024。这里只实现 forward。